# Using the library

In this tutorial, we will show how to use the library to load the data and make some customized plots. We first import the relevant functions and variables from the library and check what limits and theories are available.

In [ ]:
from eor_limits import (
    KNOWN_LIMITS,
    KNOWN_THEORIES,
    load_limit_data,
    plot_vs_k,
    plot_vs_z,
)

# KNOWN_LIMITS and KNOWN_THEORIES are both dictionaries with the keys being the names
# of the limits and theories and the values being the corresponding to its file paths
# (The paths are mostly for internal use but provide transparency on data storage)

print("* Available limits:", list(KNOWN_LIMITS.keys()))
print("* Available theories:", list(KNOWN_THEORIES.keys()))

## Looking at the data

Let's import the latest HERA limits and see what they look like! The limits are stored as a `DataSet` object, which contains both the metadata — telescope, author, year and DOI — and a `Data` object with the following attributes:
- `z`: a 1D numpy array of the redshift bins.
- `k`: a tuple of 1D numpy arrays of the $k$-bins for each redshift bin.
- `delta_squared`: a tuple of 1D numpy arrays of the $\Delta_{21}^2$ limits for each redshift bin.

alongside other attributes such as `z_lower`, `z_upper`, `k_lower`, `k_upper` for the bin edges (if available). 

A useful method of the `Data` object is `as_pandas_df()`, which converts the data into a pandas DataFrame for nicer display. The rows of the pandas DataFrame are indexed by the redshift bins, and each contain the $k$-bins and $\Delta_{21}^2$ limits for that redshift bin.

In [ ]:
hera2026 = load_limit_data("HERA2026")  # or eor_limits.DataSet.load("HERA2026")
print("Telescope:", hera2026.telescope)
print("Author:", hera2026.author)
print("Year:", hera2026.year)
print("DOI:", hera2026.doi)
print("Data:")
hera2026.data.as_pandas_df()

## Slicing the data

The library allows you to do cuts on the data, which can be quite useful for comparing datasets. There are six main functions:
- `select_closest_z(z_target)`: selects the redshift bin closest to the input $z$.
- `select_closest_k(k_target)`: selects the $k$-bin closest to the input $k$.
- `select_z_range(z_min, z_max)`: selects the redshift bins in the range $z_\mathrm{min} \leq z \leq z_\mathrm{max}$.
- `select_k_range(k_min, k_max)`: selects the $k$-bins in the range $k_\mathrm{min} \leq k \leq k_\mathrm{max}$.
- `select_delta_squared_range(delta_squared_min, delta_squared_max)`: selects the limits in the range $\Delta_{21,\mathrm{min}}^2 \leq \Delta_{21}^2 \leq \Delta_{21,\mathrm{max}}^2$.
- `select_lowest_delta_squared(per_z=True, per_tag=False)`: selects the lowest limits. `per_z` and `per_tag` are boolean flags that control how the lowest limits are selected (i.e. across redshift bins and/or across tags).

Let's do some cuts on the data to select only the limits in the range $z=7-10$ and $k=0.1-1$ h/Mpc.

In [ ]:
hera2026_trunc = hera2026.select_z_range(7, 10).select_k_range(0.1, 1)
hera2026_trunc.data.as_pandas_df()

Similarly, we can try selecting the redshift bin closest to $z=7$ and, separately, the $k$-bin closest to $k=0.1$ h/Mpc.

In [ ]:
hera2026_z7 = hera2026.select_closest_z(7)
hera2026_z7.data.as_pandas_df()

In [ ]:
hera2026_k01 = hera2026.select_closest_k(0.1)
hera2026_k01.data.as_pandas_df()

Finally, let's select the lowest limits, first per redshift bin (`per_z=True, per_tag=False` which is default), and then across all redshift bins (`per_z=False, per_tag=False`).

In [ ]:
hera2026_lowest = hera2026.select_lowest_delta_squared()
hera2026_lowest.data.as_pandas_df()

In [ ]:
hera2026_lowest = hera2026.select_lowest_delta_squared(per_z=False, per_tag=False)
hera2026_lowest.data.as_pandas_df()

## In-built plotting

### Plot vs k

Let's plot the HERA limits, including ome theoretical models for comparison, using the in-built plotting function `plot_vs_k()`. The function includes various options for slicing and styling, and color-codes the limits by redshift bin. For example:

In [ ]:
fig = plot_vs_k(
    limits=["HERA2022", "HERA2023", "HERA2026"],
    bold_limits=["HERA2026"],
    shade_limits=True,
    base_limit_style={"linewidth": 5, "shade_alpha": 0.1},
    limit_styles={"HERA2023": {"shade_alpha": 0.25, "shade_color": "C1"}},
    theories=["Mesinger2016Faint", "Mesinger2016Bright"],
    shade_theories=True,
    base_theory_style={"linestyle": "-."},
    theory_styles={
        "Mesinger2016Faint": {"color": "C3", "shade_alpha": 0.5, "shade_color": "C3"},
        "Mesinger2016Bright": {"color": "C4", "shade_alpha": 0.1, "shade_color": "C4"},
    },
    z_range=(7, 8),
    fontsize=20,
    fig_ratio=0.5,
)
fig.show()

### Plot vs redshift

Similarly, the in-built function `plot_vs_z()` allows you to plot the lowest limits (across $k$-bins) as a function of redshift. The function includes similar options for slicing and styling, and color-codes the limits by year of publication. For example:

In [ ]:
fig = plot_vs_z(
    fig_width=20.0,
    fontsize=20,
    legend_ncols=3,
    theories=["Mesinger2016Faint", "Mesinger2016Bright"],
    z_range=(5.5, 15),
    limit_styles={
        "HERA2022": {"s": 500},
        "HERA2023": {"s": 500},
        "HERA2026": {"s": 500},
    },
)
fig.show()

You can even combine the limit styling with custom legends to accentuate a particular set of observations.

In [ ]:
fig = plot_vs_z(
    fig_width=20.0,
    fontsize=20,
    legend_ncols=2,
    theories=["Mesinger2016Faint", "Mesinger2016Bright"],
    z_range=(5.5, 15),
    base_limit_style={"s": 200, "marker": "o", "color": "gray"},
    limit_styles={
        "Nunhokee2025": {"s": 300, "linewidth": 5, "marker": "s", "color": "magenta"},
        "Trott2025": {"s": 300, "linewidth": 5, "marker": "s", "color": "blue"},
    },
    legend_labeler={
        "Nunhokee2025": "Nunhokee et al. 2025",
        "Trott2025": "Trott et al. 2025",
        "Mesinger2016Faint": "Mesinger et al. 2016 faint",
        "Mesinger2016Bright": "Mesinger et al. 2016 bright",
    },
)
fig.show()